# Decidir contra alguien: minimax y poda alfa-beta

**Facsímil 2 · Inteligencia clásica** — capítulo 11 (juegos: decidir con otros actores).

Hasta ahora has buscado en mundos «pasivos»: un mapa, un sudoku, unos bloques que no te llevan la
contraria. Pero muchísimas decisiones reales se toman **contra alguien** que responde: un rival, un
mercado, otro agente. La teoría que gobierna eso es la **teoría de juegos**, y su algoritmo
fundamental es **minimax**. En este cuaderno construyes un jugador de tres en raya que **no pierde
nunca** —no por trucos, sino porque calcula todas las jugadas suyas y del rival hasta el final,
suponiendo que el otro juega lo mejor posible—. Luego le añades la **poda alfa-beta**, que descarta
ramas inútiles, y mides cuánto trabajo se ahorra sin cambiar ni una decisión.

### Qué vas a aprender
- Qué es un **juego de suma cero con información perfecta** y por qué se puede resolver «pensando hasta
  el final».
- Cómo funciona **minimax**: maximizar lo mío suponiendo que el rival minimiza.
- Qué significa el **valor de un juego** (y por qué el del tres en raya es 0: empate con juego
  perfecto).
- Cómo la **poda alfa-beta** llega a la misma decisión mirando una fracción del árbol, y por qué el
  **orden** en que exploras cambia cuánto poda.
- Por qué en juegos grandes (ajedrez, Go) hay que **cortar y estimar**, lo que enlaza con el
  aprendizaje por refuerzo.

### Cuánto cuesta
Unos 12 minutos. CPU, solo Python estándar.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [1]:
import math, random

GANAN = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
def ganador(t):
    for a,b,c in GANAN:
        if t[a] and t[a]==t[b]==t[c]: return t[a]
    return None
def lleno(t): return all(t)
def pinta(t):
    s = [c if c else str(i+1) for i,c in enumerate(t)]
    print("\n".join(" ".join(s[i:i+3]) for i in (0,3,6)))
print("Tablero vacio (los numeros marcan las casillas libres):")
pinta([""]*9)


Tablero vacio (los numeros marcan las casillas libres):
1 2 3
4 5 6
7 8 9


## 1. Qué tipo de juego es el tres en raya

El tres en raya tiene tres propiedades que lo hacen «resoluble» por completo:

- **Suma cero:** lo que uno gana, el otro lo pierde. No hay cooperación posible.
- **Información perfecta:** ambos ven el tablero entero; no hay cartas ocultas ni azar.
- **Finito:** las partidas terminan en a lo sumo 9 jugadas.

Cuando se cumplen estas tres, existe una estrategia óptima que se puede *calcular* explorando el árbol
completo de partidas. La pieza que lo hace es minimax.


## 2. Minimax: pensar hasta el final asumiendo lo peor del rival

`X` quiere ganar (**maximiza**), `O` quiere que `X` pierda (**minimiza**). El algoritmo explora el
árbol de todas las partidas posibles. En las hojas (partidas terminadas) asigna un valor: `+1` si gana
`X`, `-1` si gana `O`, `0` si hay empate. Y hacia arriba: en un turno de `X` se queda con el **máximo**
de los hijos (la mejor jugada para `X`); en un turno de `O`, con el **mínimo** (la peor para `X`,
porque `O` juega óptimo). Así, cada estado recibe el valor que tendría *si ambos jugaran perfecto* a
partir de ahí. Contamos cuántos estados visita.


In [2]:
nodos = [0]
def minimax(t, jugador):
    nodos[0] += 1
    g = ganador(t)
    if g == "X": return 1
    if g == "O": return -1
    if lleno(t): return 0
    libres = [i for i in range(9) if not t[i]]
    if jugador == "X":
        return max(minimax(t[:i]+["X"]+t[i+1:], "O") for i in libres)
    else:
        return min(minimax(t[:i]+["O"]+t[i+1:], "X") for i in libres)

nodos[0] = 0
valor = minimax([""]*9, "X")
print(f"Valor del juego desde cero: {valor}  (0 = con juego perfecto, siempre empate)")
print(f"Estados visitados por minimax puro: {nodos[0]:,}")


Valor del juego desde cero: 0  (0 = con juego perfecto, siempre empate)
Estados visitados por minimax puro: 549,946


**Lo que dice ese 0.** El tres en raya, jugado perfecto por ambos, **siempre acaba en empate**:
nadie puede forzar una victoria. Minimax no «cree» que ganará; calcula que lo mejor alcanzable contra
un rival perfecto es no perder. Y para saberlo ha tenido que mirar **más de medio millón** de estados.
Ahí está el problema que viene a resolver la poda.


## 3. Poda alfa-beta: dejar de mirar lo que ya no puede importar

La intuición es de sentido común. Imagina que estás evaluando una jugada y, a mitad, descubres que el
rival tiene una respuesta que la hace **peor** que otra jugada que ya tenías guardada. ¿Para qué seguir
mirando esa rama? La vas a descartar igual. La poda alfa-beta formaliza esto con dos cotas, **α** (lo
mejor que `X` tiene asegurado hasta ahora) y **β** (lo mejor que `O` tiene asegurado): cuando se cruzan
($\beta \le \alpha$), se **corta** la rama. Da **exactamente la misma decisión** que minimax, mirando
una fracción.


In [3]:
nodos_ab = [0]
def alfabeta(t, jugador, alfa, beta):
    nodos_ab[0] += 1
    g = ganador(t)
    if g == "X": return 1
    if g == "O": return -1
    if lleno(t): return 0
    libres = [i for i in range(9) if not t[i]]
    if jugador == "X":
        valor = -math.inf
        for i in libres:
            valor = max(valor, alfabeta(t[:i]+["X"]+t[i+1:], "O", alfa, beta))
            alfa = max(alfa, valor)
            if beta <= alfa: break        # poda: O no permitira que lleguemos aqui
        return valor
    else:
        valor = math.inf
        for i in libres:
            valor = min(valor, alfabeta(t[:i]+["O"]+t[i+1:], "X", alfa, beta))
            beta = min(beta, valor)
            if beta <= alfa: break
        return valor

nodos_ab[0] = 0
valor_ab = alfabeta([""]*9, "X", -math.inf, math.inf)
print(f"Misma decision (valor {valor_ab}), pero estados visitados: {nodos_ab[0]:,}")
print(f"\nMinimax puro:   {nodos[0]:>7,} estados")
print(f"Con poda a-b:   {nodos_ab[0]:>7,} estados")
print(f"Ahorro: {100*(1-nodos_ab[0]/nodos[0]):.0f}% del arbol, sin cambiar ni una jugada.")


Misma decision (valor 0), pero estados visitados: 18,297

Minimax puro:   549,946 estados
Con poda a-b:    18,297 estados
Ahorro: 97% del arbol, sin cambiar ni una jugada.


## 4. El orden importa: la poda premia mirar primero lo bueno

La poda alfa-beta es más potente cuanto **antes** encuentres las buenas jugadas, porque así las cotas
α y β se estrechan pronto y cortan más ramas. En el mejor caso, la poda permite explorar el doble de
profundidad con el mismo esfuerzo; en el peor (orden pésimo), apenas poda. Lo medimos: barajamos el
orden en que se prueban las jugadas y vemos cómo cambia el número de estados.


In [4]:
def alfabeta_orden(t, jugador, alfa, beta, contador, baraja):
    contador[0] += 1
    g = ganador(t)
    if g == "X": return 1
    if g == "O": return -1
    if lleno(t): return 0
    libres = [i for i in range(9) if not t[i]]
    if baraja: random.shuffle(libres)
    if jugador == "X":
        valor = -math.inf
        for i in libres:
            valor = max(valor, alfabeta_orden(t[:i]+["X"]+t[i+1:], "O", alfa, beta, contador, baraja))
            alfa = max(alfa, valor)
            if beta <= alfa: break
        return valor
    else:
        valor = math.inf
        for i in libres:
            valor = min(valor, alfabeta_orden(t[:i]+["O"]+t[i+1:], "X", alfa, beta, contador, baraja))
            beta = min(beta, valor)
            if beta <= alfa: break
        return valor

random.seed(0)
orden_fijo = [0]; alfabeta_orden([""]*9, "X", -math.inf, math.inf, orden_fijo, baraja=False)
muestras = []
for _ in range(20):
    c = [0]; alfabeta_orden([""]*9, "X", -math.inf, math.inf, c, baraja=True); muestras.append(c[0])
print(f"Sin poda (minimax):        {nodos[0]:>7,} estados")
print(f"Poda, orden fijo:          {orden_fijo[0]:>7,} estados")
print(f"Poda, orden aleatorio:     entre {min(muestras):,} y {max(muestras):,} (media {sum(muestras)//len(muestras):,})")
print("\nLa poda siempre ayuda, pero CUANTO ayuda depende del orden en que explores las jugadas.")


Sin poda (minimax):        549,946 estados
Poda, orden fijo:           18,297 estados
Poda, orden aleatorio:     entre 12,003 y 17,425 (media 14,518)

La poda siempre ayuda, pero CUANTO ayuda depende del orden en que explores las jugadas.


## 5. ¿De verdad no pierde? Que juegue 2000 partidas contra el azar

La prueba definitiva: ponemos a nuestro `X` (que decide con alfa-beta) a jugar contra un `O` que mueve
al azar. Como `X` juega perfecto, **no debería perder ni una sola** vez.


In [5]:
def mejor_jugada(t, jugador):
    libres = [i for i in range(9) if not t[i]]
    puntua = [(alfabeta(t[:i]+[jugador]+t[i+1:], "O" if jugador=="X" else "X", -math.inf, math.inf), i)
              for i in libres]
    return (max if jugador=="X" else min)(puntua)[1]

random.seed(1)
res = {"gana X": 0, "empate": 0, "gana O": 0}
for _ in range(2000):
    t = [""]*9; turno = "X"
    while ganador(t) is None and not lleno(t):
        i = mejor_jugada(t, "X") if turno == "X" else random.choice([k for k in range(9) if not t[k]])
        t[i] = turno; turno = "O" if turno == "X" else "X"
    g = ganador(t)
    res["gana X" if g=="X" else "gana O" if g=="O" else "empate"] += 1
print("Resultado de 2000 partidas (nuestro minimax = X, rival al azar = O):")
for k, v in res.items(): print(f"  {k}: {v}")
print(f"\nDerrotas de nuestro jugador: {res['gana O']}.  Jugar perfecto se nota.")


Resultado de 2000 partidas (nuestro minimax = X, rival al azar = O):
  gana X: 1988
  empate: 12
  gana O: 0

Derrotas de nuestro jugador: 0.  Jugar perfecto se nota.


## 6. Por qué el ajedrez no se resuelve así

Si esto funciona tan bien, ¿por qué no resolvemos el ajedrez igual? Porque el árbol de juego del
ajedrez es **astronómico** (más posiciones que átomos en el universo observable). Ni con poda se puede
explorar entero. La solución, desde Shannon, es **cortar a cierta profundidad** y, en vez de jugar
hasta el final, **estimar** quién va mejor con una *función de evaluación* (material, posición, rey
seguro...). Esa es la heurística del capítulo 11. Y cuando ni siquiera sabemos diseñar una buena
función de evaluación, entra en juego **aprender** a evaluar a base de jugar millones de partidas: eso
es el aprendizaje por refuerzo (facsímil 10), el camino de AlphaZero.


## 7. Pruébalo tú

1. **Juega tú contra él:** escribe un bucle que pida tu casilla con `input()` y deje responder a
   `mejor_jugada`. No vas a ganarle; como mucho, empatar.
2. **Mide el tiempo** de una partida completa con y sin poda. La decisión es idéntica; el coste, no.
3. **Cambia el tamaño** a un 4×4 (cuatro en raya): verás que el árbol crece tanto que explorarlo entero
   deja de ser viable, y entiendes por qué hace falta cortar y estimar.
4. **Un rival imperfecto:** haz que `O` juegue con alfa-beta el 80% de las veces y al azar el 20%.
   ¿Empieza a ganar `X`? Minimax explota los errores del rival.


## 8. Errores comunes

- **Olvidar que minimax asume un rival perfecto.** Contra un rival que se equivoca, minimax sigue
  siendo seguro (no pierde), pero existen estrategias que *explotan* mejor los errores.
- **Creer que la poda cambia la decisión.** No: da exactamente el mismo valor que minimax. Solo cambia
  cuánto trabajo cuesta.
- **Ignorar el orden de jugadas.** En juegos grandes, ordenar bien las jugadas (probar primero las
  prometedoras) es lo que hace la poda realmente eficaz.
- **Intentar explorar el árbol entero de un juego grande.** Imposible. Hay que cortar a una profundidad
  y estimar con una función de evaluación.


## 9. Qué te llevas

- **Minimax** decide asumiendo que el otro juega lo mejor posible: por eso su jugador no pierde. El
  valor `0` del tres en raya demuestra que, bien jugado, no hay victoria que forzar.
- **La poda alfa-beta** llega a la misma decisión visitando una parte del árbol; descartar lo que ya no
  puede cambiar el resultado es de las ideas más rentables de la IA clásica, y su eficacia depende del
  **orden** de exploración.
- Cuando el árbol es gigante (ajedrez, Go) no se explora entero: se **corta y se estima**. Ahí entran
  las heurísticas... y, más tarde, el aprendizaje por refuerzo (facsímil 10).

**Para seguir:** el facsímil 10 cambia «explorar el árbol» por «aprender a decidir jugando»; y la idea
de modelar al otro actor reaparece en agentes que negocian o compiten (facsímil 5).


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*